# ML-04 - Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NameRectified/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** - each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**What one row means:** One row = one page (content_hash_id) for one client, summarized over a calendar month.

**Which table(s):** fact_content_daily_performance (daily GSC + GA4 data) + dim_content (page metadata: content_type, intent, keyword fields).

**Time window:** Feature window = the 60 days before March 2026 (Jan 1 - Feb 28). Label window = March 2026 itself. The decision moment is March 1 - at that point you know the previous 60 days and you're ranking pages for persistent opportunity.

**Label / proxy:** persistent_ctr_gap - a page scores high if its CTR was below its position-tier median in BOTH the feature window AND March. This flags pages where underperformance is persistent, not a one-month noise blip.

**One excluded thing:** trend_direction and trend_pct - they compare past impressions to newer impressions. That answers a different question (trend decline), not this lane's question (opportunity scoring).

In [2]:
import os, getpass, duckdb, pandas as pd, numpy as np
from pathlib import Path

env_path = Path('../../.env')
if env_path.exists():
    for line in env_path.read_text().strip().split('\n'):
        if '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token: ')

con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

n_total, n_unique = con.sql(f"""
    SELECT COUNT(*), COUNT(DISTINCT content_hash_id) FROM {DIM_CONTENT}
""").fetchone()
print(f'dim_content: {n_total:,} rows, {n_unique:,} unique content_hash_ids')
print(f'Grain holds (one row per page): {n_total == n_unique}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

dim_content: 519,606 rows, 519,606 unique content_hash_ids
Grain holds (one row per page): True


## 2. Fields: feature / label / context / excluded

| Bucket | Columns | Why |
|---|---|---|
| **Context** | content_hash_id, client_hash_id | IDs for joining and splitting only - never features |
| **Feature** | gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions (from the feature window) + content_type, main_intent, search_volume, competition, cpc, word_count, content_age_days (from dim_content) | All known at decision time - describe past performance and static page properties |
| **Label / proxy** | persistent_ctr_gap (defined in section 1) | The target - flags pages where the opportunity was real and persistent |
| **Excluded** | trend_direction, trend_pct | Belong to the original trend-decline task, not this lane |
| **Excluded** | ga4_data_available, gsc_data_available | Measurement flags - use for filtering only, never as features |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a - Grain
The daily fact has one row per (content_hash_id, report_date). After aggregation to monthly, one row per content item.

### 3b - Slice count + date span
How many pages have enough data (known position + minimum impressions) in March 2026.

### 3c - Availability (IS TRUE)
How many rows have both GSC and GA4 measurement active.

### 3d - Five features for the lane
Built from the 60-day feature window, with an "available when?" line per feature.

### 3e - The leakage trap
Add one label-derived column, watch precision@50 jump, then delete it.

In [3]:
dups = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) as c
    FROM {FACT} WHERE month='2026-03'
    GROUP BY content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").fetchall()
print(f'3a - Duplicate (content_hash_id, report_date) rows: {len(dups)}')
print(f'  Grain holds: {len(dups) == 0}')

slice_info = con.sql(f"""
    SELECT COUNT(DISTINCT f.content_hash_id) as pages,
           MIN(f.report_date) as first_date,
           MAX(f.report_date) as last_date
    FROM {FACT} f
    WHERE f.month = '2026-03'
      AND f.gsc_avg_position > 0
      AND f.gsc_impressions >= 100
""").fetchone()
print(f'\n3b - Lane slice (position known + >=100 impressions)')
print(f'  Pages: {slice_info[0]:,}')
print(f'  Date range: {slice_info[1]} to {slice_info[2]}')

total_rows = con.sql(f"SELECT COUNT(*) FROM {FACT} WHERE month='2026-03'").fetchone()[0]
avail_rows = con.sql(f"""
    SELECT COUNT(*) FROM {FACT}
    WHERE month='2026-03' AND gsc_data_available IS TRUE AND ga4_data_available IS TRUE
""").fetchone()[0]
print(f'\n3c - Availability (month=2026-03)')
print(f'  Total rows: {total_rows:,}')
print(f'  Both GSC+GA4 available: {avail_rows:,} ({100*avail_rows/total_rows:.1f}%)')

features = con.sql(f"""
    SELECT f.content_hash_id,
           SUM(f.gsc_impressions) AS impressions_fw,
           AVG(f.gsc_avg_position) AS avg_pos_fw,
           STDDEV_SAMP(f.gsc_avg_position) AS pos_volatility_fw,
           c.content_type,
           c.main_intent
    FROM {FACT} f
    JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= '2026-01-01' AND f.report_date < '2026-03-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, c.content_type, c.main_intent
    HAVING SUM(f.gsc_impressions) >= 100
""").df()

print(f'\n3d - Feature frame: {len(features):,} rows x {len(features.columns)} columns')
print('\nAvailable when? lines for each feature:')
print('  1. impressions_fw - knowable at decision moment because GSC impression counts from the previous 60 days are already recorded by March 1.')
print('  2. avg_pos_fw - knowable because daily GSC positions from the previous 60 days are already recorded.')
print('  3. pos_volatility_fw - knowable because position stddev is computed from those same already-recorded daily positions.')
print('  4. content_type - knowable because this is static page metadata from dim_content, set when the page was created.')
print('  5. main_intent - knowable because this is also static page metadata from dim_content.')
features.head()

print('\n3e - Leakage trap: add a label-derived column on purpose')

ctr_data = con.sql(f"""
    SELECT f.content_hash_id,
           SUM(f.gsc_impressions) AS imp_fw,
           SUM(f.gsc_clicks) AS clk_fw,
           AVG(f.gsc_avg_position) AS pos_fw,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_impressions ELSE 0 END) AS imp_label,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_clicks ELSE 0 END) AS clk_label
    FROM {FACT} f
    WHERE f.report_date >= '2026-01-01' AND f.report_date < '2026-04-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id
    HAVING imp_fw >= 100 AND imp_label >= 100
""").df()

def assign_tier(pos):
    if pos <= 3: return 'top_3'
    if pos <= 10: return 'page_1'
    if pos <= 20: return 'striking'
    if pos <= 50: return 'page_3_5'
    return 'deep'

ctr_data['tier_fw'] = ctr_data['pos_fw'].apply(assign_tier)
tier_med = ctr_data.groupby('tier_fw').apply(
    lambda g: g['clk_fw'].sum() / g['imp_fw'].sum() * 100 if g['imp_fw'].sum() > 0 else 0
)
ctr_data['tier_median_ctr'] = ctr_data['tier_fw'].map(tier_med)
ctr_data['ctr_fw'] = ctr_data['clk_fw'] / ctr_data['imp_fw'] * 100
ctr_data['ctr_label'] = ctr_data['clk_label'] / ctr_data['imp_label'] * 100

ctr_data['gap_fw'] = ctr_data['tier_median_ctr'] - ctr_data['ctr_fw']
ctr_data['gap_label'] = ctr_data['tier_median_ctr'] - ctr_data['ctr_label']
ctr_data['label'] = ((ctr_data['gap_fw'] > 0.1) & (ctr_data['gap_label'] > 0.1)).astype(int)

base_rate = ctr_data['label'].mean()
print(f'  Base rate (persistent gap): {base_rate:.1%}')

honest = ctr_data['gap_fw'].clip(lower=0)
top_honest = honest.nlargest(50).index if len(honest) >= 50 else honest.nlargest(len(honest)).index
p_at_k_honest = ctr_data['label'].loc[top_honest].mean()
print(f'  Honest score precision@50: {p_at_k_honest:.1%}')

leaky = ctr_data['gap_label'].clip(lower=0)  
top_leaky = leaky.nlargest(50).index if len(leaky) >= 50 else leaky.nlargest(len(leaky)).index
p_at_k_leaky = ctr_data['label'].loc[top_leaky].mean()
print(f'  WITH leaked feature precision@50: {p_at_k_leaky:.1%}')
print(f'  The leaked column uses label-window data - it sees the answer. Deleting it now.')

ctr_data = ctr_data.drop(columns=['gap_label', 'label', 'tier_fw', 'tier_median_ctr',
                                  'ctr_fw', 'ctr_label', 'gap_fw'])
print(f'  Leaked columns removed. Honest score is the one that counts.')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

3a - Duplicate (content_hash_id, report_date) rows: 0
  Grain holds: True


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


3b - Lane slice (position known + >=100 impressions)
  Pages: 40,206
  Date range: 2026-03-01 to 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


3d - Feature frame: 90,235 rows x 6 columns

Available when? lines for each feature:
  1. impressions_fw - knowable at decision moment because GSC impression counts from the previous 60 days are already recorded by March 1.
  2. avg_pos_fw - knowable because daily GSC positions from the previous 60 days are already recorded.
  3. pos_volatility_fw - knowable because position stddev is computed from those same already-recorded daily positions.
  4. content_type - knowable because this is static page metadata from dim_content, set when the page was created.
  5. main_intent - knowable because this is also static page metadata from dim_content.

3e - Leakage trap: add a label-derived column on purpose


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Base rate (persistent gap): 49.5%
  Honest score precision@50: 100.0%
  WITH leaked feature precision@50: 98.0%
  The leaked column uses label-window data - it sees the answer. Deleting it now.
  Leaked columns removed. Honest score is the one that counts.


/var/folders/yy/t95v8nfd5mg826zr_b6pmwf00000gp/T/ipykernel_70214/3798982246.py:83: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tier_med = ctr_data.groupby('tier_fw').apply(


## 4. Data limits

**One named limitation: fixed calendar window, not per-client.**

The feature window (Jan 1 - Feb 28) is the same for all pages. But clients started tracking at different dates - some have no GSC or GA4 data in early January. This means pages from younger clients get filtered out or have shorter history. A production contract would check dim_clients.gsc_data_start and ga4_data_start and adjust each page's feature window to start when measurement actually began.

**What this data can never tell you:**
- **Causal claims**: I can rank pages by opportunity, but I cannot prove that editing them will improve CTR - that needs an experiment.
- **Single snapshot patterns**: One month captures only short-term patterns. Seasonal or weekly effects are invisible.

In [4]:
clients = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {DIM_CLIENTS}
    ORDER BY gsc_data_start NULLS LAST
""").df()

late_clients = (clients['gsc_data_start'] > '2026-01-01').sum() if clients['gsc_data_start'].notna().any() else 0
print(f'Clients with gsc_data_start after Jan 1, 2026: {late_clients}')
print('These clients would miss part of the feature window with a fixed calendar approach.')
clients.head(8)


Clients with gsc_data_start after Jan 1, 2026: 27
These clients would miss part of the feature window with a fixed calendar approach.


,client_hash_id,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,2025-02-11,2026-03-24
3,client_fef1a8f436438636,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,2025-06-07,NaT
5,client_b10cb2997d0c7c86,2025-06-18,2025-11-15
6,client_c182d11e4862a37d,2025-06-21,2026-02-20
7,client_65de48885f4ef01b,2025-06-21,2026-02-19


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under work/notebooks/ - then submit your repo URL on the card. Done.